In [ ]:
# Google Colab : télécharger les données de cette séance / download this session's data
import os, urllib.request
BASE = "https://raw.githubusercontent.com/Marzoug-Nabil/Nabil/main/academic/teaching/fsci6222/materials/s03/"
for f in ['immobilier_brut.csv', 'meteo_moncton_2024.csv', 'quartiers.csv']:
    os.makedirs('data', exist_ok=True)
    for dst in (f, os.path.join('data', f)):
        if not os.path.exists(dst):
            urllib.request.urlretrieve(BASE + f, dst)


# FSCI 6222 — Analyse de données
## Séance 3 : Nettoyage de données et Visualisation

**Enseignant :** Nabil Marzoug  
**Date :** 14 mai 2026  
**Université de Moncton — Faculté des sciences**

---

### Objectifs de la séance
1. Identifier les problèmes courants dans les données brutes
2. Maîtriser les techniques de nettoyage avec pandas
3. Fusionner des DataFrames (merge/join)
4. Créer des graphiques avec Matplotlib
5. Utiliser Seaborn pour la visualisation statistique
6. Mener une analyse exploratoire des données (EDA)

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("Librairies chargées avec succès!")

---
# PARTIE A — NETTOYAGE DE DONNÉES

Les données réelles sont rarement propres. Avant toute analyse, il faut les nettoyer. Voici les problèmes les plus courants :

| Problème | Exemple |
|---|---|
| Valeurs manquantes | NaN, cellules vides |
| Doublons | Lignes identiques répétées |
| Types incorrects | "deux" au lieu de 2, dates en texte |
| Incohérences textuelles | "Moncton", "moncton", "MONCTON" |
| Valeurs aberrantes | Prix de 1 850 000$ pour un condo |
| Valeurs négatives impossibles | Superficie de -85 m² |

## 1. Chargement et première exploration

In [ ]:
# Charger le dataset brut (données immobilières)
df = pd.read_csv("data/immobilier_brut.csv")

print(f"Dimensions : {df.shape}")
print(f"Colonnes : {df.columns.tolist()}")
df.head(10)

In [ ]:
# Résumé rapide
df.info()

In [ ]:
# Statistiques descriptives
df.describe()

On remarque déjà plusieurs problèmes :
- Des valeurs manquantes (Non-Null Count < 25 pour certaines colonnes)
- `nb_salles_bain` est de type `object` (devrait être numérique)
- Le prix max est 1 850 000$ (aberrant?)
- La superficie min est -85 m² (impossible)

## 2. Valeurs manquantes (NaN)

In [ ]:
# Compter les NaN par colonne
print("Valeurs manquantes par colonne :")
print(df.isnull().sum())

print(f"\nTotal de NaN : {df.isnull().sum().sum()}")
print(f"\nPourcentage de NaN par colonne :")
print((df.isnull().sum() / len(df) * 100).round(1))

In [ ]:
# Visualiser les NaN
plt.figure(figsize=(12, 4))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='YlOrRd')
plt.title('Carte des valeurs manquantes (jaune = manquant)')
plt.tight_layout()
plt.show()

In [ ]:
# Voir les lignes avec des NaN
lignes_nan = df[df.isnull().any(axis=1)]
print(f"Lignes avec au moins un NaN : {len(lignes_nan)}")
lignes_nan

### Stratégies pour traiter les NaN

| Stratégie | Quand l'utiliser | Code |
|---|---|---|
| **Supprimer les lignes** | Peu de NaN, données suffisantes | `df.dropna()` |
| **Remplir par la moyenne** | Variable numérique, distribution symétrique | `df.fillna(df.mean())` |
| **Remplir par la médiane** | Variable numérique, distribution asymétrique | `df.fillna(df.median())` |
| **Remplir par le mode** | Variable catégorielle | `df.fillna(df.mode()[0])` |
| **Remplir par une valeur fixe** | Sens métier (ex: 0, "Inconnu") | `df.fillna(0)` |

In [ ]:
# Appliquer les stratégies
df_clean = df.copy()

# Prix manquant → médiane des prix
prix_median = df_clean['prix'].median()
df_clean['prix'] = df_clean['prix'].fillna(prix_median)
print(f"Prix manquant rempli par la médiane : {prix_median}")

# Nombre de chambres manquant → médiane
chambre_median = df_clean['nb_chambres'].median()
df_clean['nb_chambres'] = df_clean['nb_chambres'].fillna(chambre_median)
print(f"Chambres manquantes remplacées par : {chambre_median}")

# Année construction manquante → médiane
annee_median = df_clean['annee_construction'].median()
df_clean['annee_construction'] = df_clean['annee_construction'].fillna(annee_median)
print(f"Année manquante remplacée par : {annee_median}")

# Date de vente manquante → "Non vendu" (propriétés en vente)
df_clean['date_vente'] = df_clean['date_vente'].fillna('Non vendu')

# Vérifier
print(f"\nNaN restants : {df_clean.isnull().sum().sum()}")

## 3. Doublons

In [ ]:
# Détecter les doublons
nb_doublons = df_clean.duplicated().sum()
print(f"Nombre de doublons : {nb_doublons}")

# Voir les doublons
doublons = df_clean[df_clean.duplicated(keep=False)]
print(f"\nLignes dupliquées :")
doublons[['id', 'adresse', 'ville', 'prix']]

In [ ]:
# Supprimer les doublons (garder la première occurrence)
avant = len(df_clean)
df_clean = df_clean.drop_duplicates()
apres = len(df_clean)
print(f"Avant : {avant} lignes")
print(f"Après : {apres} lignes")
print(f"Supprimées : {avant - apres} lignes")

In [ ]:
# Doublons sur un sous-ensemble de colonnes (même adresse et ville)
doublons_adresse = df_clean.duplicated(subset=['adresse', 'ville'], keep=False)
print(f"Adresses en double : {doublons_adresse.sum()}")
df_clean[doublons_adresse][['id', 'adresse', 'ville', 'prix']]

In [ ]:
import pandas as pd

df_1 = pd.DataFrame({
    "nom": ["Ali", "Sara", "Ali", "Omar", "Sara"],
    "note": [80, 90, 80, 70, 90]
})

## 4. Types de données incorrects

In [ ]:
# Vérifier les types
print(df_clean.dtypes)
print("\nProblème : nb_salles_bain est 'object' au lieu de numérique")
print("Valeurs uniques :", df_clean['nb_salles_bain'].unique())

In [ ]:
# Corriger nb_salles_bain : remplacer "deux" par 2
df_clean['nb_salles_bain'] = df_clean['nb_salles_bain'].replace('deux', '2')

# Convertir en numérique
df_clean['nb_salles_bain'] = pd.to_numeric(df_clean['nb_salles_bain'])
print(f"Type après correction : {df_clean['nb_salles_bain'].dtype}")
print(f"Valeurs uniques : {sorted(df_clean['nb_salles_bain'].unique())}")

In [ ]:
# Corriger les dates
print("Dates avant correction :")
print(df_clean['date_vente'].unique())

# Convertir en datetime (ignorer les erreurs pour les formats invalides)
df_clean['date_vente'] = pd.to_datetime(df_clean['date_vente'], errors='coerce', dayfirst=False)

# Vérifier les NaT (dates non converties)
print(f"\nDates non converties (NaT) : {df_clean['date_vente'].isna().sum()}")
print(f"Type : {df_clean['date_vente'].dtype}")

## 5. Incohérences textuelles

In [ ]:
# Vérifier les valeurs uniques des colonnes textuelles
print("Province :", df_clean['province'].unique())
print("Type :", df_clean['type'].unique())
print("Statut :", df_clean['statut'].unique())

In [ ]:
# Standardiser : tout en majuscule pour province, capitalize pour type et statut
df_clean['province'] = df_clean['province'].str.upper()
df_clean['type'] = df_clean['type'].str.capitalize()
df_clean['statut'] = df_clean['statut'].str.capitalize()

print("Après correction :")
print("Province :", df_clean['province'].unique())
print("Type :", df_clean['type'].unique())
print("Statut :", df_clean['statut'].unique())

## 6. Valeurs aberrantes (outliers)

In [ ]:
# Détecter les valeurs aberrantes avec des boxplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].boxplot(df_clean['prix'].dropna())
axes[0].set_title('Prix')
axes[0].set_ylabel('$')

axes[1].boxplot(df_clean['superficie_m2'].dropna())
axes[1].set_title('Superficie (m²)')

axes[2].boxplot(df_clean['nb_chambres'].dropna())
axes[2].set_title('Nb chambres')

plt.suptitle('Détection des valeurs aberrantes', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Détecter avec le z-score
from scipy import stats

z_prix = np.abs(stats.zscore(df_clean['prix'].dropna()))
aberrants_prix = df_clean[z_prix > 2]
print(f"Prix aberrants (|z| > 2) :")
print(aberrants_prix[['adresse', 'ville', 'prix', 'type']])

In [ ]:
# Corriger les valeurs impossibles
# Superficie négative
print(f"Superficies négatives : {(df_clean['superficie_m2'] < 0).sum()}")
df_clean.loc[df_clean['superficie_m2'] < 0, 'superficie_m2'] = np.nan
df_clean['superficie_m2'] = df_clean['superficie_m2'].fillna(df_clean['superficie_m2'].median())

# Prix aberrant (1.85M pour une maison de 130m²) → probablement 185000
df_clean.loc[df_clean['prix'] > 1000000, 'prix'] = df_clean.loc[df_clean['prix'] > 1000000, 'prix'] / 10

print(f"Prix max après correction : {df_clean['prix'].max()}")
print(f"Superficie min après correction : {df_clean['superficie_m2'].min()}")

### Exercice 1 — Nettoyage de données

En utilisant le DataFrame `df_clean` actuel :

1. Vérifier qu'il ne reste plus de NaN, de doublons, ni de valeurs aberrantes
2. Ajouter une colonne `prix_m2` = prix / superficie_m2
3. Ajouter une colonne `age_maison` = 2026 - annee_construction
4. Afficher un résumé avec `describe()` pour vérifier que les statistiques sont cohérentes
5. Exporter le DataFrame nettoyé en CSV : `data/immobilier_propre.csv`

In [ ]:
# Votre code ici


<details>
<summary> Cliquer pour voir la solution</summary>

```python
# 1. Vérification
print(f"NaN restants : {df_clean.isnull().sum().sum()}")
print(f"Doublons : {df_clean.duplicated().sum()}")
print(f"Prix négatifs : {(df_clean['prix'] < 0).sum()}")
print(f"Superficie négative : {(df_clean['superficie_m2'] < 0).sum()}")

# 2. Prix au m²
df_clean['prix_m2'] = (df_clean['prix'] / df_clean['superficie_m2']).round(2)

# 3. Âge de la maison
df_clean['age_maison'] = 2026 - df_clean['annee_construction']

# 4. Résumé
print(df_clean.describe().round(2))

# 5. Exporter
df_clean.to_csv('data/immobilier_propre.csv', index=False)
print("Fichier exporté!")
```

</details>

## 7. Fusionner des DataFrames (merge)

In [ ]:
# Charger le dataset des quartiers
df_quartiers = pd.read_csv("data/quartiers.csv")
print("Dataset quartiers :")
df_quartiers

### Types de merge

| Type | Description | Résultat |
|---|---|---|
| `inner` | Garde seulement les correspondances | Intersection |
| `left` | Garde toutes les lignes de gauche | Gauche complète |
| `right` | Garde toutes les lignes de droite | Droite complète |
| `outer` | Garde tout | Union |

In [ ]:
# Inner merge : seulement les villes présentes dans les deux tables
df_merged = pd.merge(df_clean, df_quartiers, on='ville', how='inner')
print(f"Inner merge : {len(df_merged)} lignes")
df_merged[['adresse', 'ville', 'prix', 'score_ecole', 'score_securite']].head()

In [ ]:
# Left merge : garder toutes les propriétés, même sans info quartier
df_left = pd.merge(df_clean, df_quartiers, on='ville', how='left')
print(f"Left merge : {len(df_left)} lignes")
print(f"NaN dans score_ecole : {df_left['score_ecole'].isna().sum()}")

In [ ]:
# Concat : empiler des DataFrames verticalement
df1 = df_clean[df_clean['ville'] == 'Moncton']
df2 = df_clean[df_clean['ville'] == 'Dieppe']

df_concat = pd.concat([df1, df2], ignore_index=True)
print(f"df1 : {len(df1)} + df2 : {len(df2)} = concat : {len(df_concat)}")

### Exercice — Nettoyage de données complet

Charger `data/immobilier_brut.csv` et appliquer toutes les étapes de nettoyage :

**Étape 1: Valeurs manquantes**
- Afficher le nombre de NaN par colonne (`isnull().sum()`)
- Remplir les prix manquants par la **médiane**
- Remplir les chambres manquantes par la **médiane**
- Remplir l'année de construction manquante par la **médiane**
- Remplir les dates de vente manquantes par `"Non vendu"`

**Étape 2: Doublons**
- Compter le nombre de doublons (`duplicated().sum()`)
- Supprimer les doublons en gardant la première occurrence

**Étape 3: Types incorrects**
- Vérifier les types avec `dtypes`
- Remplacer `"deux"` par `"2"` dans `nb_salles_bain`
- Convertir `nb_salles_bain` en numérique (`pd.to_numeric()`)
- Convertir `date_vente` en datetime (`pd.to_datetime()`)

**Étape 4: Incohérences textuelles**
- Standardiser `province` en majuscules (`.str.upper()`)
- Standardiser `type` et `statut` avec `.str.capitalize()`

**Étape 5: Valeurs aberrantes**
- Remplacer les superficies négatives par `NaN`, puis remplir par la médiane
- Corriger le prix extrême (1 850 000$ → probablement 185 000$)

**Étape 6: Colonnes calculées**
- Ajouter `prix_m2` = prix / superficie_m2
- Ajouter `age_maison` = 2026 - annee_construction

**Étape 7: Fusion**
- Charger `data/quartiers.csv`
- Fusionner avec un `left` merge sur la colonne `ville`

**Étape 8: Vérification et export**
- Vérifier : `isnull().sum().sum()` doit être 0 (sauf pour les colonnes du merge)
- Vérifier : `duplicated().sum()` doit être 0
- Afficher `describe()` pour vérifier la cohérence
- Exporter en CSV : `data/immobilier_propre.csv`

<details>
<summary> Cliquer pour voir la solution</summary>

```python
import pandas as pd
import numpy as np

# ============================================
# Étape 1 — Valeurs manquantes
# ============================================
df = pd.read_csv("data/immobilier_brut.csv")
print(f"Dimensions initiales : {df.shape}")
print(f"\nValeurs manquantes :\n{df.isnull().sum()}")

# Remplir par la médiane
df['prix'] = df['prix'].fillna(df['prix'].median())
df['nb_chambres'] = df['nb_chambres'].fillna(df['nb_chambres'].median())
df['annee_construction'] = df['annee_construction'].fillna(df['annee_construction'].median())
df['date_vente'] = df['date_vente'].fillna('Non vendu')

print(f"\nNaN après traitement : {df.isnull().sum().sum()}")

# ============================================
# Étape 2 — Doublons
# ============================================
print(f"\nDoublons avant : {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Doublons après : {df.duplicated().sum()}")
print(f"Lignes restantes : {len(df)}")

# ============================================
# Étape 3 — Types incorrects
# ============================================
# nb_salles_bain : "deux" → "2" puis conversion
df['nb_salles_bain'] = df['nb_salles_bain'].replace('deux', '2')
df['nb_salles_bain'] = pd.to_numeric(df['nb_salles_bain'])

# date_vente : convertir en datetime
df['date_vente'] = pd.to_datetime(df['date_vente'], errors='coerce')

print(f"\nTypes après correction :")
print(df[['nb_salles_bain', 'date_vente']].dtypes)

# ============================================
# Étape 4 — Incohérences textuelles
# ============================================
df['province'] = df['province'].str.upper()
df['type'] = df['type'].str.capitalize()
df['statut'] = df['statut'].str.capitalize()

print(f"\nProvince : {df['province'].unique()}")
print(f"Type     : {df['type'].unique()}")
print(f"Statut   : {df['statut'].unique()}")

# ============================================
# Étape 5 — Valeurs aberrantes
# ============================================
# Superficie négative → NaN → médiane
print(f"\nSuperficies négatives : {(df['superficie_m2'] < 0).sum()}")
df.loc[df['superficie_m2'] < 0, 'superficie_m2'] = np.nan
df['superficie_m2'] = df['superficie_m2'].fillna(df['superficie_m2'].median())

# Prix extrême (> 1M) → diviser par 10
print(f"Prix > 1M : {(df['prix'] > 1000000).sum()}")
df.loc[df['prix'] > 1000000, 'prix'] = df.loc[df['prix'] > 1000000, 'prix'] / 10

print(f"Prix max après correction : {df['prix'].max()}")
print(f"Superficie min après correction : {df['superficie_m2'].min()}")

# ============================================
# Étape 6 — Colonnes calculées
# ============================================
df['prix_m2'] = (df['prix'] / df['superficie_m2']).round(2)
df['age_maison'] = 2026 - df['annee_construction']

print(f"\nNouvelles colonnes :")
print(df[['prix', 'superficie_m2', 'prix_m2', 'age_maison']].head())

# ============================================
# Étape 7 — Fusion
# ============================================
df_quartiers = pd.read_csv("data/quartiers.csv")
df = pd.merge(df, df_quartiers, on='ville', how='left')

print(f"\nAprès merge : {df.shape}")
print(f"Colonnes : {df.columns.tolist()}")

# ============================================
# Étape 8 — Vérification et export
# ============================================
print(f"\n=== Vérification finale ===")
print(f"Doublons : {df.duplicated().sum()}")
print(f"NaN par colonne :\n{df.isnull().sum()}")
print(f"\nStatistiques :")
print(df.describe().round(2))

# Exporter
df.to_csv('data/immobilier_propre.csv', index=False)
print(f"\nFichier exporté : data/immobilier_propre.csv ({len(df)} lignes)")
```

</details>

---
# PARTIE B — VISUALISATION

La visualisation est essentielle pour :
- **Explorer** les données (trouver des patterns)
- **Communiquer** les résultats (raconter une histoire)
- **Détecter** les anomalies (valeurs aberrantes, erreurs)

Deux librairies principales :
- **Matplotlib** : la base, contrôle total, graphiques personnalisés
- **Seaborn** : basé sur Matplotlib, plus beau par défaut, orienté statistiques

## 8. Matplotlib — Les bases

In [ ]:
# Charger les données météo pour la visualisation
df_meteo = pd.read_csv("meteo_moncton_2024.csv")
df_meteo

### 8.1 Graphique en ligne (line plot)

In [ ]:
# Graphique simple
plt.figure(figsize=(12, 5))
plt.plot(df_meteo['mois'], df_meteo['temperature_moy'], marker='o', color='steelblue', linewidth=2)
plt.title('Température moyenne à Moncton (2024)')
plt.xlabel('Mois')
plt.ylabel('Température (°C)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plusieurs courbes sur le même graphique
plt.figure(figsize=(12, 5))
plt.plot(df_meteo['mois'], df_meteo['temperature_max'], 'r-o', label='Max', linewidth=2)
plt.plot(df_meteo['mois'], df_meteo['temperature_moy'], 'b-s', label='Moyenne', linewidth=2)
plt.plot(df_meteo['mois'], df_meteo['temperature_min'], 'g-^', label='Min', linewidth=2)

# Zone entre min et max
plt.fill_between(range(12), df_meteo['temperature_min'], df_meteo['temperature_max'], alpha=0.1, color='blue')

plt.axhline(y=0, color='black', linestyle='--', alpha=0.3)
plt.title('Températures à Moncton (2024)')
plt.xlabel('Mois')
plt.ylabel('Température (°C)')
plt.xticks(range(12), df_meteo['mois'], rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

### 8.2 Diagramme en barres (bar plot)

In [ ]:
# Barres verticales
plt.figure(figsize=(12, 5))
colors = ['steelblue' if p < 100 else 'coral' for p in df_meteo['precipitation_mm']]
plt.bar(df_meteo['mois'], df_meteo['precipitation_mm'], color=colors, edgecolor='white')
plt.axhline(y=df_meteo['precipitation_mm'].mean(), color='red', linestyle='--', 
            label=f'Moyenne = {df_meteo["precipitation_mm"].mean():.0f} mm')
plt.title('Précipitations mensuelles à Moncton (2024)')
plt.xlabel('Mois')
plt.ylabel('Précipitations (mm)')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Barres groupées : prix moyen par ville et type (immobilier)
df_immo = pd.read_csv("data/immobilier_brut.csv")
prix_ville_type = df_immo.groupby(['ville', 'type'])['prix'].mean().unstack()

prix_ville_type.plot(kind='bar', figsize=(10, 5), colormap='Set2', edgecolor='white')
plt.title('Prix moyen par ville et type de propriété')
plt.xlabel('Ville')
plt.ylabel('Prix moyen ($)')
plt.xticks(rotation=0)
plt.legend(title='Type')
plt.tight_layout()
plt.show()

### 8.3 Histogramme

In [ ]:
# Distribution des prix
plt.figure(figsize=(10, 5))
plt.hist(df_clean['prix'], bins=15, color='steelblue', edgecolor='white', alpha=0.7)
plt.axvline(x=df_clean['prix'].mean(), color='red', linestyle='--', linewidth=2, label=f'Moyenne = {df_clean["prix"].mean():,.0f}$')
plt.axvline(x=df_clean['prix'].median(), color='green', linestyle='--', linewidth=2, label=f'Médiane = {df_clean["prix"].median():,.0f}$')
plt.title('Distribution des prix immobiliers')
plt.xlabel('Prix ($)')
plt.ylabel('Fréquence')
plt.legend()
plt.tight_layout()
plt.show()

### 8.4 Nuage de points (scatter plot)

In [ ]:
# Relation entre superficie et prix
plt.figure(figsize=(10, 6))
plt.scatter(df_clean['superficie_m2'], df_clean['prix'], 
            c=df_clean['nb_chambres'], cmap='viridis', s=80, alpha=0.7, edgecolors='white')
plt.colorbar(label='Nb chambres')
plt.title('Prix vs Superficie')
plt.xlabel('Superficie (m²)')
plt.ylabel('Prix ($)')
plt.tight_layout()
plt.show()

### 8.5 Diagramme circulaire (pie chart)

In [ ]:
# Répartition par ville
repartition = df_clean['ville'].value_counts()

plt.figure(figsize=(8, 8))
plt.pie(repartition, labels=repartition.index, autopct='%1.1f%%', 
        colors=sns.color_palette('Set2'), startangle=90, textprops={'fontsize': 12})
plt.title('Répartition des propriétés par ville')
plt.tight_layout()
plt.show()

### 8.6 Subplots — Plusieurs graphiques dans une figure

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Histogramme des prix
axes[0, 0].hist(df_clean['prix'], bins=12, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Distribution des prix')
axes[0, 0].set_xlabel('Prix ($)')

# 2. Barres : nb propriétés par ville
df_clean['ville'].value_counts().plot(kind='bar', ax=axes[0, 1], color='coral', edgecolor='white')
axes[0, 1].set_title('Propriétés par ville')
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=0)

# 3. Scatter : prix vs superficie
axes[1, 0].scatter(df_clean['superficie_m2'], df_clean['prix'], color='green', alpha=0.6)
axes[1, 0].set_title('Prix vs Superficie')
axes[1, 0].set_xlabel('Superficie (m²)')
axes[1, 0].set_ylabel('Prix ($)')

# 4. Boxplot des prix par type
df_clean.boxplot(column='prix', by='type', ax=axes[1, 1])
axes[1, 1].set_title('Prix par type')
axes[1, 1].set_xlabel('Type')
axes[1, 1].set_ylabel('Prix ($)')
plt.suptitle('')  # Enlever le titre auto du boxplot

plt.suptitle('Tableau de bord immobilier', fontsize=16)
plt.tight_layout()
plt.show()

### Exercice 2 — Matplotlib

En utilisant le dataset `data/meteo_moncton_2024.csv` :

1. Créer un graphique en barres de l'ensoleillement par mois (heures)
2. Créer un scatter plot : ensoleillement vs température moyenne. Ajouter un titre et des labels
3. Créer un subplot 2×2 avec :
   - Température moyenne (line plot)
   - Précipitations (bar plot)
   - Humidité (line plot)
   - Vent (bar plot)
4. Sauvegarder la figure : `plt.savefig('meteo_dashboard.png', dpi=150)`

In [ ]:
# Votre code ici


<details>
<summary> Cliquer pour voir la solution</summary>

```python
df_m = pd.read_csv('data/meteo_moncton_2024.csv')

# 1. Ensoleillement
plt.figure(figsize=(12, 5))
plt.bar(df_m['mois'], df_m['ensoleillement_h'], color='gold', edgecolor='white')
plt.title('Ensoleillement mensuel à Moncton (2024)')
plt.xlabel('Mois')
plt.ylabel('Heures d\'ensoleillement')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# 2. Scatter
plt.figure(figsize=(8, 6))
plt.scatter(df_m['ensoleillement_h'], df_m['temperature_moy'], s=100, c='coral')
for i, mois in enumerate(df_m['mois']):
    plt.annotate(mois[:3], (df_m['ensoleillement_h'].iloc[i], df_m['temperature_moy'].iloc[i]),
                 fontsize=8, ha='left')
plt.xlabel('Ensoleillement (h)')
plt.ylabel('Température moyenne (°C)')
plt.title('Ensoleillement vs Température')
plt.tight_layout()
plt.show()

# 3-4. Dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].plot(df_m['mois'], df_m['temperature_moy'], 'ro-')
axes[0,0].set_title('Température')
axes[0,0].tick_params(axis='x', rotation=45)

axes[0,1].bar(df_m['mois'], df_m['precipitation_mm'], color='steelblue')
axes[0,1].set_title('Précipitations')
axes[0,1].tick_params(axis='x', rotation=45)

axes[1,0].plot(df_m['mois'], df_m['humidite_pct'], 'gs-')
axes[1,0].set_title('Humidité')
axes[1,0].tick_params(axis='x', rotation=45)

axes[1,1].bar(df_m['mois'], df_m['vent_kmh'], color='orange')
axes[1,1].set_title('Vent')
axes[1,1].tick_params(axis='x', rotation=45)

plt.suptitle('Dashboard Météo Moncton 2024', fontsize=16)
plt.tight_layout()
plt.savefig('meteo_dashboard.png', dpi=150)
plt.show()
print('Figure sauvegardée!')
```

</details>

## 9. Seaborn — Visualisation statistique

### 9.1 Graphiques de distribution

In [ ]:
# histplot : histogramme + courbe de densité
plt.figure(figsize=(10, 5))
sns.histplot(data=df_clean, x='prix', kde=True, bins=15, color='steelblue')
plt.title('Distribution des prix (histogramme + KDE)')
plt.xlabel('Prix ($)')
plt.tight_layout()
plt.show()

In [ ]:
# kdeplot : courbes de densité comparatives
plt.figure(figsize=(10, 5))
for ville in df_clean['ville'].unique():
    sns.kdeplot(data=df_clean[df_clean['ville'] == ville], x='prix', label=ville, fill=True, alpha=0.3)
plt.title('Distribution des prix par ville')
plt.xlabel('Prix ($)')
plt.legend()
plt.tight_layout()
plt.show()

### 9.2 Graphiques catégoriels

In [ ]:
# boxplot : distribution par catégorie
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_clean, x='ville', y='prix', palette='Set2')
plt.title('Distribution des prix par ville')
plt.ylabel('Prix ($)')
plt.tight_layout()
plt.show()

In [ ]:
# violinplot : boxplot + densité
plt.figure(figsize=(10, 5))
sns.violinplot(data=df_clean, x='ville', y='prix', palette='Set2', inner='quartile')
plt.title('Distribution des prix par ville (violin plot)')
plt.ylabel('Prix ($)')
plt.tight_layout()
plt.show()

In [ ]:
# barplot : moyenne avec intervalle de confiance
plt.figure(figsize=(10, 5))
sns.barplot(data=df_clean, x='ville', y='prix', hue='type', palette='Set2', ci=95)
plt.title('Prix moyen par ville et type (avec IC 95%)')
plt.ylabel('Prix moyen ($)')
plt.tight_layout()
plt.show()

In [ ]:
# countplot : compter les occurrences
plt.figure(figsize=(10, 5))
sns.countplot(data=df_clean, x='ville', hue='type', palette='Set2')
plt.title('Nombre de propriétés par ville et type')
plt.ylabel('Nombre')
plt.tight_layout()
plt.show()

### 9.3 Graphiques relationnels

In [ ]:
# scatterplot avec hue
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_clean, x='superficie_m2', y='prix', hue='ville', 
                style='type', s=100, palette='Set2')
plt.title('Prix vs Superficie (par ville et type)')
plt.xlabel('Superficie (m²)')
plt.ylabel('Prix ($)')
plt.tight_layout()
plt.show()

In [ ]:
# regplot : scatter + droite de régression
plt.figure(figsize=(10, 6))
sns.regplot(data=df_clean, x='superficie_m2', y='prix', 
            scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})
plt.title('Prix vs Superficie (avec régression)')
plt.xlabel('Superficie (m²)')
plt.ylabel('Prix ($)')
plt.tight_layout()
plt.show()

### 9.4 Heatmap de corrélation

In [ ]:
# Matrice de corrélation
colonnes_num = ['prix', 'nb_chambres', 'nb_salles_bain', 'superficie_m2', 'annee_construction']
corr = df_clean[colonnes_num].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1)
plt.title('Matrice de corrélation')
plt.tight_layout()
plt.show()

### 9.5 pairplot : toutes les relations en un coup d'œil

In [ ]:
# Pairplot (peut prendre quelques secondes)
sns.pairplot(df_clean[['prix', 'superficie_m2', 'nb_chambres', 'type']], 
             hue='type', palette='Set2', diag_kind='kde')
plt.suptitle('Pairplot des variables numériques', y=1.02)
plt.show()

### Exercice 3 — Seaborn

En utilisant le dataset immobilier nettoyé :

1. Créer un `violinplot` des prix par ville, avec `hue='type'`
2. Créer un `jointplot` (scatter + distributions marginales) de prix vs superficie
   (`sns.jointplot(data=df_clean, x='superficie_m2', y='prix', kind='reg')`)
3. Créer une heatmap de corrélation incluant la colonne `prix_m2` et `age_maison`
4. Créer un `catplot` montrant la distribution des prix par ville (type = 'box') avec des facettes par type de propriété

In [ ]:
# Votre code ici


<details>
<summary> Cliquer pour voir la solution</summary>

```python
# 1. Violinplot
plt.figure(figsize=(12, 5))
sns.violinplot(data=df_clean, x='ville', y='prix', hue='type', palette='Set2', split=True)
plt.title('Distribution des prix par ville et type')
plt.ylabel('Prix ($)')
plt.tight_layout()
plt.show()

# 2. Jointplot
sns.jointplot(data=df_clean, x='superficie_m2', y='prix', kind='reg', height=8, color='steelblue')
plt.suptitle('Prix vs Superficie (jointplot)', y=1.02)
plt.show()

# 3. Heatmap avec nouvelles colonnes
cols = ['prix', 'superficie_m2', 'nb_chambres', 'nb_salles_bain', 'prix_m2', 'age_maison']
corr = df_clean[cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Matrice de corrélation étendue')
plt.tight_layout()
plt.show()

# 4. Catplot
sns.catplot(data=df_clean, x='ville', y='prix', col='type', kind='box', palette='Set2', height=5)
plt.suptitle('Prix par ville (facettes par type)', y=1.02)
plt.show()
```

</details>

## 10. Choisir le bon graphique

| Question | Type de graphique | Code |
|---|---|---|
| Comment une variable évolue-t-elle? | Line plot | `plt.plot()` |
| Quelle est la distribution? | Histogramme / KDE | `sns.histplot()`, `sns.kdeplot()` |
| Comparer des catégories? | Bar plot / Box plot | `sns.barplot()`, `sns.boxplot()` |
| Quelle relation entre 2 variables? | Scatter plot | `sns.scatterplot()`, `sns.regplot()` |
| Quelle proportion? | Pie chart | `plt.pie()` |
| Quelles corrélations? | Heatmap | `sns.heatmap()` |
| Vue d'ensemble? | Pairplot / Subplots | `sns.pairplot()`, `plt.subplots()` |

---
# Partie 11 — Exercice récapitulatif : EDA complète

### Exercice 4 — Analyse exploratoire complète

Réaliser une EDA complète du dataset immobilier en suivant ces étapes :

1. **Nettoyage** : charger `data/immobilier_brut.csv`, identifier et corriger tous les problèmes
2. **Enrichissement** : ajouter les colonnes `prix_m2`, `age_maison`, fusionner avec `data/quartiers.csv`
3. **Analyse univariée** : distribution de chaque variable numérique (histogrammes)
4. **Analyse bivariée** : relations entre variables (scatter, boxplot, heatmap de corrélation)
5. **Analyse par catégorie** : prix moyen par ville, par type, par tranche d'âge de maison
6. **Dashboard final** : créer une figure 2×3 avec les 6 graphiques les plus informatifs
7. **Conclusions** : résumer les insights principaux en 3-5 points

In [ ]:
# 1. Nettoyage


In [ ]:
# 2. Enrichissement


In [ ]:
# 3. Analyse univariée


In [ ]:
# 4. Analyse bivariée


In [ ]:
# 5. Analyse par catégorie


In [ ]:
# 6. Dashboard final


### 7. Conclusions

Écrire vos insights ici :

1. ...
2. ...
3. ...

<details>
<summary> Cliquer pour voir la solution</summary>

```python
# 1. Nettoyage complet
df = pd.read_csv('data/immobilier_brut.csv')
df = df.drop_duplicates()
df['prix'] = df['prix'].fillna(df['prix'].median())
df['nb_chambres'] = df['nb_chambres'].fillna(df['nb_chambres'].median())
df['annee_construction'] = df['annee_construction'].fillna(df['annee_construction'].median())
df['date_vente'] = df['date_vente'].fillna('Non vendu')
df['nb_salles_bain'] = df['nb_salles_bain'].replace('deux', '2')
df['nb_salles_bain'] = pd.to_numeric(df['nb_salles_bain'])
df['province'] = df['province'].str.upper()
df['type'] = df['type'].str.capitalize()
df['statut'] = df['statut'].str.capitalize()
df.loc[df['superficie_m2'] < 0, 'superficie_m2'] = df['superficie_m2'].median()
df.loc[df['prix'] > 1000000, 'prix'] = df.loc[df['prix'] > 1000000, 'prix'] / 10

# 2. Enrichissement
df['prix_m2'] = (df['prix'] / df['superficie_m2']).round(2)
df['age_maison'] = 2026 - df['annee_construction']
quartiers = pd.read_csv('data/quartiers.csv')
df = pd.merge(df, quartiers, on='ville', how='left')

# 3. Analyse univariée
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, col in zip(axes.flat, ['prix', 'superficie_m2', 'nb_chambres', 'age_maison', 'prix_m2', 'nb_salles_bain']):
    sns.histplot(data=df, x=col, kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribution de {col}')
plt.tight_layout()
plt.show()

# 4. Analyse bivariée
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(data=df, x='superficie_m2', y='prix', hue='ville', ax=axes[0])
sns.boxplot(data=df, x='ville', y='prix', ax=axes[1], palette='Set2')
corr = df[['prix', 'superficie_m2', 'nb_chambres', 'age_maison', 'prix_m2']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[2])
plt.tight_layout()
plt.show()

# 5. Analyse par catégorie
print(df.groupby('ville')['prix'].agg(['mean', 'median', 'count']).round(0))
print(df.groupby('type')['prix'].agg(['mean', 'median', 'count']).round(0))

# 6. Dashboard
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
sns.histplot(data=df, x='prix', kde=True, ax=axes[0,0])
axes[0,0].set_title('Distribution des prix')
sns.boxplot(data=df, x='ville', y='prix', ax=axes[0,1], palette='Set2')
axes[0,1].set_title('Prix par ville')
sns.scatterplot(data=df, x='superficie_m2', y='prix', hue='type', ax=axes[0,2])
axes[0,2].set_title('Prix vs Superficie')
sns.barplot(data=df, x='ville', y='prix_m2', ax=axes[1,0], palette='Set2')
axes[1,0].set_title('Prix/m² par ville')
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1,1])
axes[1,1].set_title('Corrélations')
df['ville'].value_counts().plot(kind='pie', ax=axes[1,2], autopct='%1.0f%%', colors=sns.color_palette('Set2'))
axes[1,2].set_title('Répartition par ville')
axes[1,2].set_ylabel('')
plt.suptitle('Dashboard EDA — Immobilier NB', fontsize=16)
plt.tight_layout()
plt.show()
```

</details>

---

## Récapitulatif

### Nettoyage

| Opération | Code |
|---|---|
| Détecter les NaN | `df.isnull().sum()` |
| Supprimer les NaN | `df.dropna()` |
| Remplir les NaN | `df.fillna(valeur)` |
| Détecter les doublons | `df.duplicated().sum()` |
| Supprimer les doublons | `df.drop_duplicates()` |
| Convertir un type | `pd.to_numeric()`, `pd.to_datetime()` |
| Standardiser le texte | `.str.upper()`, `.str.capitalize()` |
| Fusionner | `pd.merge(df1, df2, on='col', how='inner')` |

### Visualisation

| Type | Matplotlib | Seaborn |
|---|---|---|
| Ligne | `plt.plot()` | `sns.lineplot()` |
| Barres | `plt.bar()` | `sns.barplot()`, `sns.countplot()` |
| Histogramme | `plt.hist()` | `sns.histplot()` |
| Scatter | `plt.scatter()` | `sns.scatterplot()`, `sns.regplot()` |
| Boxplot | `plt.boxplot()` | `sns.boxplot()` |
| Heatmap | — | `sns.heatmap()` |
| Pairplot | — | `sns.pairplot()` |

---

## Prochaine séance

**Séance 4 — Statistiques descriptives**
- Tendance centrale (moyenne, médiane, mode)
- Dispersion (variance, écart-type, étendue)
- Forme (asymétrie, aplatissement)
- Quartiles et boxplots

**Date :** Mardi 19 mai 2026, 18h30

---

*FSCI 6222 — Université de Moncton — Été 2026*